# Evaluation Summary

Audits and visualizes performance, timing, memory, and hardware artifacts produced by the evaluation workflow.

## Environment

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Evaluation Configuration

In [ ]:
from src.evaluate import load_selected_evaluation_configuration
from src.evaluate.summary_utils import discover_sensitivity_models
from src.evaluate.judge_utils import resolve_evaluation_selection

experiment_config, selected_judge_config = load_selected_evaluation_configuration()
eval_type = experiment_config["eval_type"]
selected_methods, selected_milestones, selected_runs = resolve_evaluation_selection(experiment_config, eval_type)
evaluation_dir = PROJECT_ROOT / "data" / "evaluation"

editing_methods = list(experiment_config.get("editing_methods", []))
model_key = experiment_config.get("model_key")
result_names = list(experiment_config.get("result_names", ["es_loc", "port"]))
judge_repeats = int(selected_judge_config["judge"]["repeats"])
show_std = False

if eval_type == "sensitivity":
    milestones = list(experiment_config["eval_milestones"])
    n_runs = int(experiment_config["n_runs"])
    sensitivity_milestone = int(milestones[-1])
    sensitivity_method = editing_methods[0]
    model_keys = discover_sensitivity_models(
        milestone=sensitivity_milestone,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        method=sensitivity_method,
    )
    methods = selected_methods
    intrinsic_study = None
elif eval_type == "intrinsic":
    intrinsic_study = str(experiment_config.get("study", "encoder"))
    intrinsic_config = experiment_config
    study_config = intrinsic_config["studies"][intrinsic_study]
    milestones = list(study_config["eval_milestones"])
    n_runs = int(study_config.get("n_runs", intrinsic_config.get("n_runs", len(study_config.get("runs", [])))))
    methods = selected_methods
    model_keys = [model_key]
    methods = selected_methods
    sensitivity_milestone = None
    sensitivity_method = None
else:
    intrinsic_study = None
    milestones = list(experiment_config["eval_milestones"])
    n_runs = int(experiment_config["n_runs"])
    model_keys = [model_key]

## Summary Helpers

In [ ]:
from IPython.display import display
from src.evaluate.summary_tables import (
    table_perf_results,
    table_time_results,
    table_composite_results,
    table_memory_results,
    table_hardware_results,
    audit_result_grid,
    table_sensitivity_runs,
    table_sensitivity_variation,
    table_sensitivity_audit,
    table_intrinsic_results,
)
from src.evaluate.summary_plots import (
    plot_perf_results,
    plot_time_results,
    plot_composite_results,
    plot_memory_results,
    plot_pareto_results,
)

## Completeness Audit

In [ ]:
if eval_type in {"output", "sensitivity"}:
    audit_result_grid(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        result_groups=["unscored", "scored", "perf", "time", "mem"],
    )

## Sensitivity Summary

This branch preserves deterministic order rows and reports variation across the matched model-size runs.

In [ ]:
if eval_type == "sensitivity":
    print(f"Sensitivity models discovered: {model_keys}")
    display(table_sensitivity_audit(
        model_keys=model_keys,
        milestone=sensitivity_milestone,
        method=sensitivity_method,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
    ))
    display(table_sensitivity_runs(
        model_keys=model_keys,
        milestone=sensitivity_milestone,
        method=sensitivity_method,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
    ))
    display(table_sensitivity_variation(
        model_keys=model_keys,
        milestone=sensitivity_milestone,
        method=sensitivity_method,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        evaluation_dir=evaluation_dir,
        show_std=show_std,
        eval_type=eval_type,
    ))

## Intrinsic Summary

This section reports the selected intrinsic study (encoder, intervention, or safety).

In [ ]:
if eval_type == "intrinsic":
    display(table_intrinsic_results(
        study=intrinsic_study,
        intrinsic_config=intrinsic_config,
        judge_repeats=judge_repeats,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
    ))

## Output Performance

In [ ]:
if eval_type == "output":
    table_perf_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

In [ ]:
if eval_type == "output":
    plot_perf_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

## Output Time

In [ ]:
if eval_type == "output":
    table_time_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

In [ ]:
if eval_type == "output":
    plot_time_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

## Output Composite

In [ ]:
if eval_type == "output":
    table_composite_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

In [ ]:
if eval_type == "output":
    plot_composite_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

## Output Memory

In [ ]:
if eval_type == "output":
    table_memory_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

In [ ]:
if eval_type == "output":
    plot_memory_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )

## Output Hardware

In [ ]:
if eval_type == "output":
    table_hardware_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=editing_methods,
        result_names=result_names,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
    )

## Output Pareto Frontier

In [ ]:
if eval_type == "output":
    plot_pareto_results(
        model_key=model_key,
        evaluation_dir=evaluation_dir,
        eval_type=eval_type,
        milestones=milestones,
        methods=methods,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
        show_std=show_std,
    )